[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/henryjhu/Country-Resolution-LLM-and-GeoNames-Database/blob/main/LLM%20Data%20Preparation.ipynb)

<Strong> <font face="Verdana" size="5"> AML Country Resolution Via LLM and GeoNames Database Search </font></strong> <br>
<b> <font face="Verdana" size="3"> Project Highlight: </font></b><br>
<font face="Verdana" size="3"> This is the exact replica of the work which was carried out on a real project at one of the banks. </font><br>
<b> <font face="Verdana" size="3"> Project Objective: </font></b><br>
<font face="Verdana" size="3"> Unstructured ISO 20022 SWIFT pacs.008 messages contain free-form address fields that may include typographical errors, incomplete address information, and missing country names for both originators and beneficiaries. These data quality issues make it challenging for banks to accurately identify the countries associated with cross-border wire transfers, potentially affecting Anti-Money Laundering (AML) compliance monitoring. The objective of this project is to leverage both Large Language Model (LLM) and GeoNames Database Searches for correctly identifying the countries of these very messy address texts. Accurate country resolution helps transaction monitoring rules and sanction screen rules correctly identify cross-border transactions. </font><br>
<b> <font face="Verdana" size="3">Data Scientist:</font></b><font face="Verdana" size="3"> Henry J. Hu</font><br>
<b> <font face="Verdana" size="3">Runtime and Inference Engine:</font></b><font face="Verdana" size="3"> Ollama</font><br>
<b> <font face="Verdana" size="3">Programming Language:</font></b><font face="Verdana" size="3"> Python</font><br>
<b> <font face="Verdana" size="3"> Data Source: </font></b><br>
<font face="Verdana" size="3"> Data Owner: </font> <a href="https://postalcodezip.com ">postalcodezip.com</a> <br>
<font face="Verdana" size="3"> Number of Countries: </font> 25 <br>
<font face="Verdana" size="3"> Number of Addresses: </font> 2290 <br>
<b> <font face="Verdana" size="3"> Data Privacy: </font></b><br>
<font face="Verdana" size="3">
Because company policy requires all employees to sign a Non-Disclosure Agreement (NDA) that prohibits the disclosure of confidential information related to the company’s data and business operations, this replica machine learning project uses only freely and publicly accessible data. The project does not disclose or contain any confidential information related to any company’s data, operations, or business practices.
</font><br>
<b> <font face="Verdana" size="3">Last Update:</font></b><font face="Verdana" size="3"> October 21st, 2026</font>

# Importing Libraries

In [126]:
import os
import sys
import pandas as pd
import numpy as np
import ollama
import json
import requests
import time
import subprocess
from google.colab import drive

# Setting Up Ollama Runtime Environment and Inference Engine

In [128]:
# This is a Llama API key acts as a digital ID, secret token or password used for authenticating the code or application when sending requests to run Meta's open-source Llama artificial intelligence models.
os.environ["OLLAMA_KEEP_ALIVE"]= '0'
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
os.environ['OLLAMA_KV_CACHE_TYPE'] = 'q8_0'  # Optional: Greatly reduces VRAM usage alongside Flash Attention
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'

In [129]:
# Increase the maximum height and width of the output area.
os.system("mode con cols=50 lines=5000")

32512

In [130]:
# Set the column width to unlimited
pd.set_option('display.max_colwidth', None)

In [131]:
!apt update

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,172 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,733 kB]
Fetched 4,058 kB in 3s (1,195 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading 

In [132]:
!apt install lspci

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package lspci


In [133]:
!apt install -y pciutils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.10.0-2build1).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.


In [134]:
!apt install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.


In [135]:
# Installing Ollama
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

Installing Ollama...


CompletedProcess(args='curl -fsSL https://ollama.com/install.sh | sh', returncode=0)

In [136]:
# Start the ollama server as a background process
os.makedirs("/content/ollama_logs", exist_ok = True)
log = open("/content/ollama_logs/server.log","w")
proc=subprocess.Popen(
	["ollama", "serve"],
	stdout=log, stderr=log,
	env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"}
)

# Wait until the server responds is up runnng
for i in range(30):
  try:
    r=requests.get("http://localhost:11434/api/tags", timeout=1)
    if r.ok:
      print("Ollama server is up!")
      break
  except Exception:
    pass
  time.sleep(5)
else:
  print("Ollama did not start in time")

Ollama server is up!


In [137]:
# Installing llama3.1:8b
print("Installing llama3.1:8b...")
subprocess.run("ollama run llama3.1:8b", shell=True)

Installing llama3.1:8b...


CompletedProcess(args='ollama run llama3.1:8b', returncode=0)

In [138]:
!ollama list

NAME           ID              SIZE      MODIFIED    
llama3.1:8b    46e0c10c039e    4.9 GB    3 hours ago    


In [39]:
!pip install -q ollama

In [58]:
!pip install openpyxl

# Mounting Google Drive

In [42]:
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


# Importing Data

In [158]:
df = pd.read_excel("gdrive/MyDrive/Github_Projects/LLM_Addresses.xlsx", index_col = None)

# Exploratory Data Analysis

In [141]:
# Find out the number of rows and columns on the dataframe
df.shape

(200, 13)

In [142]:
# Find out the data type of each column on the dataframe
df.dtypes

,0
index,int64
MessageType,object
CountryCodeVerified,object
Address,object
Number,object
Street,object
Region,object
City,object
State,object
Province,object


In [143]:
# Find out the number of null values in each column on the dataframe
df.isnull().sum()

,0
index,0
MessageType,0
CountryCodeVerified,0
Address,0
Number,89
Street,77
Region,66
City,76
State,137
Province,152


In [144]:
# Examine the first 5 rows of the dataframe
df.head(n=5)

,index,MessageType,CountryCodeVerified,Address,Number,Street,Region,City,State,Province,PostalCode,CountryCodeLLM,ConfidenceScore
0,2178,pacs.008,GB,"Moor Lane, England, Willoughby , , , CV23 8BT",CV23 8BT,Moor Lane,England,Willoughby,NaN,NaN,CV23 8BT,GB,Our analysis suggests that the provided address is a UK address.
1,1271,pacs.008,NG,"Tolupe St. Ikeja Lagos State , 100263",100263,Tolupe St.,Ikeja,Lagos,Lagos State,NaN,100263,NG,0.9
2,1268,pacs.008,NG,"Omolola St., , Lagos Lagos State, , 101213",NaN,Omolola St.,Lagos,Lagos,Lagos State,NaN,101213,NG,class 2
3,44,pacs.008,AR,"Calle Bolivia, Buenos Aires Moreno B1744 CFI",Calle Bolivia,Bolivia,Buenos Aires,Moreno,NaN,NaN,B1744,CFI,?
4,2025,pacs.008,AE,"شارع 8, أبوظبي العين, , , 30055",8,شارع 8,أبوظبي,العين,NaN,NaN,30055,AE,default


In [145]:
# Retrieve the last 5 rows of the dataframe
df.tail(n=5)

,index,MessageType,CountryCodeVerified,Address,Number,Street,Region,City,State,Province,PostalCode,CountryCodeLLM,ConfidenceScore
195,332,pacs.008,CL,"General Salvo, Región Metropolitana de Santiago, Providencia, , , 7500713",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,1290,pacs.008,NG,"Uden St. , Awka, Anambra State, , , 420231",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,69,pacs.008,AR,"Calle Lumiere, Córdoba Cordoba , X5020 FCB",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,530,pacs.008,GE,"შ 109 Imereti Sochkheti, ,",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
199,1380,pacs.008,RU,"Stroitelnaya ul, Hakasiya Resp, Fyrkal s, , , 655232",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Phase 1: Extract Address Components
<b> <font face="Verdana" size="3"> Objective: </font></b><br>
The first objective is to take advantage of Llama's ability to infer from the address components in the text for determinting the country.<br>
The second objective is to take advantage of Llama's ability to extract address components from very messy international addresses.

<b> <font face="Verdana" size="3"> Model Information: </font></b><br>
<font face="Verdana" size="3"> Model Developer:</font> Meta<br>
<font face="Verdana" size="3"> Model Version:</font> llama3.1:8b<br>
<font face="Verdana" size="3">Model Page:</font>
https://huggingface.co/meta-llama/Llama-3.1-8B
<br>
<font face="Verdana" size="3">Model Description:</font>
The Meta Llama 3.1 collection of multilingual large language models (LLMs) is a collection of pretrained and instruction tuned generative models in 8B, 70B and 405B sizes (text in/text out). The Llama 3.1 instruction tuned text only models (8B, 70B, 405B) are optimized for multilingual dialogue use cases and outperform many of the available open source and closed chat models on common industry benchmarks.


In [113]:
##############################################################################################
# Purpose:
# This function is used for extracting individual address components from a text string.
##############################################################################################

def AddressParser (df):
    # Define the JSON schema for the address components
    schema = {
        "type": "object",
        "properties": {
            "Number": {"type": "string"},
            "Street": {"type": "string"},
            "Region": {"type": "string"},
            "City": {"type": "string"},
            "State": {"type": "string"},
            "Province": {"type": "string"},
            "District": {"type": "string"},
            "Postal Code": {"type": "string"},
            "Inferred Country ISO2 Code":  {"type": "string"},
            "Confidence Score" :   {"type": "string"},
        },
        "required": ['Number', 'Street', 'Region', 'City', 'State', 'Province', 'District', 'Postal Code','Inferred Country ISO2 Code','Confidence Score'],
    }

    # Record the process start time
    start_time = time.perf_counter()

    for row in df.itertuples(index=True):
        # Access attributes using dot notation (.Column_Name)
        print(f"Index: {row.Index}, Address: {row.Address}")

        inputTxt = row.Address
        replacements = {"'": " ",'"': " ","\n": " ","\\n": " ","\t": " ","\\": " "}
        for old, new in replacements.items():
            inputTxt = inputTxt.replace(old, new)

        #inputTxt = "Aden - Mukalla Raod# Abyan Governorate@ حورة: . . : "
        prompt = "Parse this address:" + inputTxt
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": "llama3.1:8b",
                    "prompt": prompt,
                    "format": schema,  # Enforces structured JSON output
                    "stream": False,
                    "keep_alive": 0,
                },
            )
            parsed_address = json.loads(response.json()["response"])
            print(json.dumps(parsed_address, indent=2))

            # put the extracted address components back into the orginal dataframe
            df.at[row.Index,'Number' ] = parsed_address['Number']
            df.at[row.Index,'Street' ] = parsed_address['Street']
            df.at[row.Index,'Region' ] = parsed_address['Region']
            df.at[row.Index,'City' ] = parsed_address['City']
            df.at[row.Index,'State' ] = parsed_address['State']
            df.at[row.Index,'Province' ] = parsed_address['Province']
            df.at[row.Index,'PostalCode' ] = parsed_address['Postal Code']
            df.at[row.Index,'CountryCodeLLM' ] = parsed_address['Inferred Country ISO2 Code']
            df.at[row.Index,'ConfidenceScore' ] = parsed_address['Confidence Score']

        except json.JSONDecodeError as e:
            print(f"JSONDecodeError: {e}")
            pass
        except ValueError as e:
            print(f"ValueError: {e}")
            pass
        except ollama.ResponseError as e:
            # Handles API or server-side errors from Ollama
            print(f"Ollama API Error: {e.error} (Status code: {e.status_code})")
            pass
        except ConnectionError as e:
            print(f"ConnectionError: {e}")
            pass
        except Exception as e:
            # Catches any unexpected general errors
            print(f"Other Exception: {e}")
            pass

    # Record the process end time
    end_time = time.perf_counter()
    # Calculate process elapsed time
    execution_time = end_time - start_time
    print(f"Execution time: {execution_time:.6f} seconds")

In [159]:
# Get a random sample of addresses for testing.
np.random.seed(42)
df1 = df.sample(n=200)
df1 = df1.reset_index()

In [160]:
df1.shape

(77, 13)

In [162]:
AddressParser (df1)

Index: 7, Address:  151 Street  Sharjah, Sharjah,     , 
{
  "Number": "151",
  "Street": "Street",
  "Region": "Sharjah",
  "City": "Sharjah",
  "State": "",
  "Province": "",
  "District": "",
  "Postal Code": "",
  "Inferred Country ISO2 Code": "AE",
  "Confidence Score": "Some"
}
Index: 24, Address:  Caserio el Rebellon, Principado de Asturias, Godella,       33758
{
  "Number": "33758",
  "Street": "Caserio el Rebellon",
  "Region": "Principado de Asturias",
  "City": "Godella",
  "State": "Asturias",
  "Province": "Asturias",
  "District": "Unknown",
  "Postal Code": "33758",
  "Inferred Country ISO2 Code": "ES",
  "Confidence Score": "1"
}
Index: 26, Address:  طريق الجبجب الثلوث, Dhamar Governorate, Al Hudayyah,       
{
  "Number": "",
  "Street": "\u0627\u0644\u062c\u0628\u062c\u0628 \u0627\u0644\u062b\u0644\u0648\u062b",
  "Region": "Dhamar Governorate",
  "City": "Al Hudayyah",
  "State": "",
  "Province": "",
  "District": "",
  "Postal Code": "",
  "Inferred Country ISO2 C

In [167]:
df1.head(n=15)

,index,MessageType,CountryCodeVerified,Address,Number,Street,Region,City,State,Province,PostalCode,CountryCodeLLM,ConfidenceScore
0,2014,pacs.008,AE,"151 Street Sharjah, Sharjah, ,",151,Street,Sharjah,Sharjah,,,,AE,Some
1,1683,pacs.008,ES,"Caserio el Rebellon, Principado de Asturias, Godella, 33758",33758,Caserio el Rebellon,Principado de Asturias,Godella,Asturias,Asturias,33758,ES,1
2,2281,pacs.008,YE,"طريق الجبجب الثلوث, Dhamar Governorate, Al Hudayyah,",,الجبجب الثلوث,Dhamar Governorate,Al Hudayyah,,,,YE,For more information visit: https://nominatim.openstreetmap.org/search?q=الجبجب%20الثلوث%20Dhamar%20Governorate%20Al%20Hudayyah&format=json&limit=1
3,1691,pacs.008,ES,"Calle Bosc, Cataluña, Rubí , , 8191",",",Calle Bosc,Cataluña,Rubí,,,8191,ES,"catalonia is not a city, but a region, and Rubi is a city in the region."
4,736,pacs.008,IL,"Nitsanim Haifa Haifa , , 34354 37",: 34354,Nitsanim,Haifa,Haifa,,,34354,IL,: 0.2
5,196,pacs.008,BY,"Пограничная полоса Border Line Grodno Region Сопоцкинский сельский Совет, , ,",Пограничная полоса,Border Line,Grodno Region,Сопоцкинский сельский Совет,,,,BY,0.8
6,482,pacs.008,CU,"Carretera Central Guantánamo, Baitiqurí, , ,",Carretera Central,Carretera Central,Guantánamo,Baitiqurí,Guantánamo,Guantánamo,,CU,Medium
7,1675,pacs.008,ES,"Diseminado Afores Cataluña, Pegueroles , , 25286",25286,Diseminado Afores,Cataluña,Pegueroles,,,25286,ES,Implicit
8,435,pacs.008,CU,"Carretera Sancti Spiritus - Yaguajay Sancti Spiritus, Jíquima de Peláez , ,",,Carretera Sancti Spiritus - Yaguajay,Sancti Spiritus,Jíquima de Peláez,,,,,}
9,555,pacs.008,GE,"ზუგდიდი-ჯვარი-მესტია-ლასდილი, Samegrelo-Upper Svaneti Zeda Etseri, , , ,","} ZUGDIDI-ჯვარი-მესტია-ლასდილი, Samegrelo-Upper Svaneti",Zeda Etseri,,Zugdidi,Samegrelo-Upper Svaneti,,,,}


# Phase 2: Validate Phase 2 Result Against GeoNames Database
<b> <font face="Verdana" size="3"> Objective: </font></b><br>
Since the precision and accuracy rates of Llama will never be 100% on messy addresses, it is neccessary to validate the result against a secondary country deciphering method. This method uses the address components parsed by the Llama model as parameters and search for the best matching country in the GeoNames database.

In [168]:
##############################################################################################
# Purpose:
# This function returns the most frequently occurring value in a one dimensional numpy array.
##############################################################################################

def MostFrequentArrItem(arr=None):
    if arr.size == 0:
        return None

    frequency_map = {}
    max_count = 0
    most_frequent_item = None

    for item in arr:
        frequency_map[item] = frequency_map.get(item, 0) + 1
        if frequency_map[item] > max_count:
            max_count = frequency_map[item]
            most_frequent_item = item

    return most_frequent_item

In [169]:
##############################################################################################
# Purpose:
# This function searches for the best match country in the GeoNames database.
# Inpute parameters:
# An array of street number, street name, region name, city name, state name, and postal code.
##############################################################################################

def GeonamesCountrySearch(INarr=None, username=None):
    # GeoNames free search API endpoint
    base_url = "http://api.geonames.org/searchJSON"
    #base_url = "http://geonames.org"

    # splitting the input array of address components into 9 search combinations
    search_query1 = INarr[0:5].astype(str)
    search_query2 = INarr[1:3].astype(str)
    search_query3 = INarr[1:4].astype(str)
    search_query4 = INarr[1:5].astype(str)
    search_query5 = INarr[2:4].astype(str)
    search_query6 = INarr[2:5].astype(str)
    search_query7 = INarr[3:5].astype(str)
    search_query8 = INarr[[1,3]].astype(str)
    search_query9 = INarr[[1,4]].astype(str)

    # producing search strings by concetenating address components with comma delimiter
    search_query1_f = ",".join(search_query1[search_query1 != ""])
    search_query2_f = ",".join(search_query2[search_query2 != ""])
    search_query3_f = ",".join(search_query3[search_query3 != ""])
    search_query4_f = ",".join(search_query4[search_query4 != ""])
    search_query5_f = ",".join(search_query5[search_query5 != ""])
    search_query6_f = ",".join(search_query6[search_query6 != ""])
    search_query7_f = ",".join(search_query7[search_query7 != ""])
    search_query8_f = ",".join(search_query8[search_query8 != ""])
    search_query9_f = ",".join(search_query9[search_query9 != ""])

    # storing the 9 search strings in an array
    SrcArr = [search_query1_f,
              search_query2_f,
              search_query3_f,
              search_query4_f,
              search_query5_f,
              search_query6_f,
              search_query7_f,
              search_query8_f,
              search_query9_f]

    # Search GeoNames for all 9 possible of address combinations.
    # Store all 90 (10 outcomes for each of the 9 searches) possible countries into an array.
    # Returns the most frequently found country.

    i=0
    my_array = np.array([])

    for item in SrcArr:
        print (f"Address Part: {item}")
        params = {
            "q": item,
            #"name_equals": item,
            "maxRows": 10,           # We only need the top/most accurate match
            "username": username,    # Your activated GeoNames username
            "fuzzy": 1
        }
        response = requests.get(base_url, params=params)
        response.raise_for_status() # Raise exception for bad status codes
        data = response.json()

        if data.get('geonames'): # Check if the API returned any results
            result = data.get('geonames')
            temp_list = []
            for geoname in result:
                i=i+1
                print(f"GeoNames Result#{i} - Address Part: {geoname.get('name')}, Country: {geoname.get('countryCode')}")
                temp_list.append(geoname.get('countryCode'))
            my_array = np.array(temp_list)

    if my_array.size == 0:
        return None
    else:
        # Returns the most frequently found country.
        return MostFrequentArrItem(my_array)

In [170]:
############################################################
# Executing Phase 2
############################################################

USERNAME = "henryjhu"
df1['CountryCodeGeoNames'] = ''
i=0
# Record the process start time
start_time = time.perf_counter()
for row in df1.itertuples(index=True):
    i=i+1
    print (f"Address#{i} Begin")
    country = GeonamesCountrySearch(np.array([row.Number, row.Street, row.City, row.State, row.Region, row.PostalCode]), USERNAME)
    if (not(country)) or (country == "None"): country = "Unknown"
    print (f"Address#{i} GeoNames Country: {country}")
    df1.at[row.Index,'CountryCodeGeoNames' ] = country
# Record the process end time
end_time = time.perf_counter()
# Calculate process elapsed time
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

Streaming output truncated to the last 5000 lines.
GeoNames Result#7 - Address Part: Main-Yankee Street Historic District, Country: US
GeoNames Result#8 - Address Part: Main Street Commercial District, Country: US
GeoNames Result#9 - Address Part: Pinson Main Street Historic District, Country: US
GeoNames Result#10 - Address Part: Main Street Historic District, Country: US
GeoNames Result#11 - Address Part: Main Street Historic District, Country: US
GeoNames Result#12 - Address Part: Fort Payne Main Street Historic District, Country: US
GeoNames Result#13 - Address Part: Main Street Historic District, Country: US
Address#35 GeoNames Country: US
Address#36 Begin
Address Part: 49beon-Gil,Jido-Ro,Todang-Dong, (no state listed),Gyeonggi
Address Part: Jido-Ro,Todang-Dong
Address Part: Jido-Ro,Todang-Dong, (no state listed)
Address Part: Jido-Ro,Todang-Dong, (no state listed),Gyeonggi
Address Part: Todang-Dong, (no state listed)
Address Part: Todang-Dong, (no state listed),Gyeonggi
Address P

In [171]:
# Examine the first 5 rows of the dataframe
df1.head(n=5)

,index,MessageType,CountryCodeVerified,Address,Number,Street,Region,City,State,Province,PostalCode,CountryCodeLLM,ConfidenceScore,CountryCodeGeoNames
0,2014,pacs.008,AE,"151 Street Sharjah, Sharjah, ,",151,Street,Sharjah,Sharjah,,,,AE,Some,LC
1,1683,pacs.008,ES,"Caserio el Rebellon, Principado de Asturias, Godella, 33758",33758,Caserio el Rebellon,Principado de Asturias,Godella,Asturias,Asturias,33758,ES,1,ES
2,2281,pacs.008,YE,"طريق الجبجب الثلوث, Dhamar Governorate, Al Hudayyah,",,الجبجب الثلوث,Dhamar Governorate,Al Hudayyah,,,,YE,For more information visit: https://nominatim.openstreetmap.org/search?q=الجبجب%20الثلوث%20Dhamar%20Governorate%20Al%20Hudayyah&format=json&limit=1,YE
3,1691,pacs.008,ES,"Calle Bosc, Cataluña, Rubí , , 8191",",",Calle Bosc,Cataluña,Rubí,,,8191,ES,"catalonia is not a city, but a region, and Rubi is a city in the region.",ES
4,736,pacs.008,IL,"Nitsanim Haifa Haifa , , 34354 37",: 34354,Nitsanim,Haifa,Haifa,,,34354,IL,: 0.2,IL


# Phase 3: Reducing The Number of Incorrect Results


In [173]:
df1['CountryCodeFinal'] = ''

# Iterate through the entire df1 dataframe
for idx, row in df1.iterrows():
    #Exracting the values of each field and convert them into strings.
    ctryLLM = str(row['CountryCodeLLM'])
    ctryGEO = str(row['CountryCodeGeoNames'])

    # remove special characters from field CountryCodeLLM
    replacements = {".": " ", "'": " ",'"': " ","\n": " ","\\n": " ","\t": " ","\\": " "}
    for old, new in replacements.items():
        ctryLLM = ctryLLM.replace(old, new).strip()

    # Logic to figure out what the final country code should be.
    if ctryLLM == ctryGEO:
        df1.iloc[idx,14] = ctryGEO
    elif (ctryLLM == 'unspecified')  & (ctryGEO == 'Unknown'):
        df1.iloc[idx,14] = 'Unknown'
    elif (ctryLLM == '(none)')  & (ctryGEO == 'Unknown'):
        df1.iloc[idx,14] = 'Unknown'
    elif (pd.isna(ctryLLM))  & (ctryGEO == 'Unknown'):
        df1.iloc[idx,14] = 'Unknown'
    elif (not pd.isna(ctryLLM))  & (ctryGEO == 'Unknown'):
        df1.iloc[idx,14] = ctryLLM
    elif (pd.isna(ctryLLM))  & (ctryGEO != 'Unknown'):
        df1.iloc[idx,14] = ctryGEO
    elif ctryLLM != ctryGEO:
        df1.iloc[idx,14] = ctryGEO
    else:
        df1.iloc[idx,14] = 'Unknown'

In [174]:
# Examine the first 5 rows of the dataframe
df1.head(n=5)

,index,MessageType,CountryCodeVerified,Address,Number,Street,Region,City,State,Province,PostalCode,CountryCodeLLM,ConfidenceScore,CountryCodeGeoNames,CountryCodeFinal
0,2014,pacs.008,AE,"151 Street Sharjah, Sharjah, ,",151,Street,Sharjah,Sharjah,,,,AE,Some,LC,LC
1,1683,pacs.008,ES,"Caserio el Rebellon, Principado de Asturias, Godella, 33758",33758,Caserio el Rebellon,Principado de Asturias,Godella,Asturias,Asturias,33758,ES,1,ES,ES
2,2281,pacs.008,YE,"طريق الجبجب الثلوث, Dhamar Governorate, Al Hudayyah,",,الجبجب الثلوث,Dhamar Governorate,Al Hudayyah,,,,YE,For more information visit: https://nominatim.openstreetmap.org/search?q=الجبجب%20الثلوث%20Dhamar%20Governorate%20Al%20Hudayyah&format=json&limit=1,YE,YE
3,1691,pacs.008,ES,"Calle Bosc, Cataluña, Rubí , , 8191",",",Calle Bosc,Cataluña,Rubí,,,8191,ES,"catalonia is not a city, but a region, and Rubi is a city in the region.",ES,ES
4,736,pacs.008,IL,"Nitsanim Haifa Haifa , , 34354 37",: 34354,Nitsanim,Haifa,Haifa,,,34354,IL,: 0.2,IL,IL


# Recall, Precision and Accuracy Rates

True Positives = Wires which the deciphered countries matched with the verified countries <br>
False Positives = Wires which the deciphered countries do not matched with the verified countries <br>
False Negatives = Wires which the deciphered countries are "Unknown".

In [175]:
TP = (df1['CountryCodeVerified'] == df1['CountryCodeFinal']).sum()
FP = (df1['CountryCodeVerified'] != df1['CountryCodeFinal']).sum()
FN = (df1['CountryCodeFinal']=="Unknown").sum()

<b><font size=3>Recall Rate</font></b>

In [176]:
TP/(TP+FN)

np.float64(1.0)

<b><font size=3>Precision Rate</font></b>

In [177]:
TP/(TP+FP)

np.float64(0.805)

<b><font size=3>Acccracy Rate</font></b>

In [178]:
TP/df1.shape[0]

np.float64(0.805)

In [179]:
# Exporting the final result to an Excel file.
df1.to_excel("gdrive/MyDrive/Github_Projects/LLM_Address_Deciphered_Result.xlsx", index=False)